# M2.3 Conductors, Induced Charge and Images

Book pointer: chapter 7, sections 7.1 (conductors), 7.2 (ideal conductor in equilibrium), 7.3 (Laplace's equation with conductors), 7.4 (imaging methods and mirror charge).

## 1. Recall

1. List the four equilibrium properties of an ideal conductor and give a one-line reason for each.
2. Why must the field just outside a conductor be normal to its surface, and what is its magnitude in terms of $\sigma$?
3. Describe the mirror-charge construction for a point charge near a grounded plane. Which two conditions does the image solution satisfy, and why is that enough?
4. For a grounded sphere of radius $R$ and a charge $q$ at distance $d$: image charge and its position.
5. Where in a relaxation solver does a conductor enter, and what boundary condition does it impose?

## 2. Principle to algorithm

A conductor in equilibrium is an equipotential, so in a relaxation solver it is a set of fixed nodes all holding the same value. Everything else follows from the solution: the field at the surface is normal and its magnitude gives the induced surface charge through $\sigma = \epsilon_0 E_n$, and the total induced charge is $\epsilon_0$ times the flux of $\mathbf{E}$ through any box enclosing the conductor.

The 2D solver models systems that are uniform along $z$, so the natural point source is a line charge $\lambda$ with potential $-(\lambda/2\pi\epsilon_0)\ln r$. Its image in a grounded plane is $-\lambda$ at the mirror position, and the exact potential is $(\lambda/2\pi\epsilon_0)\ln(r_-/r_+)$. We fix the outer boundary of the grid to that exact value, put the line charge in as a single-cell source with $\rho h^2 = \lambda$, and let the solver produce the interior. The uniqueness theorem says it must match the image solution.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, relax
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)

In [ ]:
lam, d = 1e-9, 0.5
n = 161
x = np.linspace(0, 2, n); y = np.linspace(-1, 1, n)
h = x[1] - x[0]
X, Y = np.meshgrid(x, y, indexing="ij")

def image_solution(X, Y):
    r_plus, r_minus = np.hypot(X - d, Y), np.hypot(X + d, Y)
    with np.errstate(divide="ignore"):
        return lam / (2 * np.pi * EPS0) * np.log(r_minus / r_plus)

exact = image_solution(X, Y)
V0 = np.where(np.isfinite(exact), exact, 0.0); V0[1:-1, 1:-1] = 0.0
fixed = np.zeros((n, n), dtype=bool)
rho = np.zeros((n, n))
i0, j0 = np.argmin(abs(x - d)), np.argmin(abs(y))
rho[i0, j0] = lam / h**2

V = relax.solve_sparse(V0, fixed, rho=rho, h=h)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
lv = np.linspace(0, exact[np.isfinite(exact)].max() * 0.4, 15)
axes[0].contour(X, Y, V, levels=lv, colors="C0")
axes[0].contour(X, Y, np.where(np.isfinite(exact), exact, np.nan), levels=lv, colors="C3", linestyles="--")
axes[0].axvline(0, color="k", lw=3); axes[0].plot(d, 0, "ro"); axes[0].set_aspect("equal")
axes[0].set_title("solver (solid) vs image solution (dashed)")
far = np.hypot(X - d, Y) > 0.1
axes[1].imshow(np.where(far, np.abs(V - exact) / np.abs(exact[far]).max(), np.nan).T, origin="lower",
               extent=(0, 2, -1, 1), cmap="magma")
axes[1].set_title("relative difference, away from the source"); plt.show()
print(f"max relative difference at least 0.1 from the line charge: {(np.abs(V - exact)[far] / np.abs(exact[far]).max()).max():.2e}")

### Induced surface charge on the plane

The plane is the set of nodes with $x = 0$. The field there is normal to the plane, so $\sigma = \epsilon_0 E_x$ with the one-sided difference $E_x \approx -(V_{1,j} - V_{0,j})/h$. The exact result for a line charge is $\sigma(y) = -\dfrac{\lambda\, d}{\pi\,(d^2 + y^2)}$ and the total induced charge per unit depth is $-\lambda$.

In [ ]:
sigma_num = -EPS0 * (V[1, :] - V[0, :]) / h
sigma_exact = -lam * d / (np.pi * (d**2 + y**2))
plt.plot(y, sigma_exact, "k-", lw=3, alpha=0.3, label="exact")
plt.plot(y, sigma_num, "C1.", ms=4, label="from solver")
plt.xlabel("y"); plt.ylabel("sigma  [C/m^2]"); plt.legend(); plt.show()
print(f"total induced charge / lambda: {np.trapezoid(sigma_num, y) / lam:+.4f}   (exact -1 on an infinite plane, here the plane spans |y| <= 1)")
print(f"exact value over the same span: {np.trapezoid(sigma_exact, y) / lam:+.4f}")

### Exercise: induced charge from a Gaussian box

Replace the body of `induced_charge_on_block` so that it returns the total induced charge per unit depth on a grounded block, computed as $\epsilon_0$ times the outward flux of $\mathbf{E}$ through a rectangle just outside the block. Use `fields.box_flux` on the gridded field. The check places the block in a uniform field between plates, where the induced charge must be zero by symmetry, and then in the field of a nearby line charge, where it must be negative.

In [ ]:
def grounded_block_potential(rho_extra=None, top_V=0.0):
    m = 121
    V0 = np.zeros((m, m)); fx = np.zeros((m, m), dtype=bool)
    xs = np.linspace(-1, 1, m); hb = xs[1] - xs[0]
    V0[:, -1] = top_V; V0[:, 0] = -top_V
    V0[:, 0] = -top_V; V0[0, :] = V0[-1, :] = np.linspace(-top_V, top_V, m)   # sides linear between the plates
    a, b = 50, 71
    V0[a:b, a:b] = 0.0; fx[a:b, a:b] = True
    return relax.solve_sparse(V0, fx, rho=rho_extra, h=hb), hb, (a, b)

def induced_charge_on_block(V, hb, block):
    """eps0 times the outward flux through a box two cells outside the block."""
    a, b = block
    E = -fields.gradient(V, hb, hb)
    return EPS0 * fields.box_flux(E, (hb, hb), (a - 2, a - 2), (b + 1, b + 1))    # replace with your own box


V_u, hb, blk = grounded_block_potential(top_V=1.0)
Q_u = induced_charge_on_block(V_u, hb, blk)
rho_line = np.zeros((121, 121)); rho_line[100, 60] = 1e-9 / hb**2
V_l, _, _ = grounded_block_potential(rho_extra=rho_line)
Q_l = induced_charge_on_block(V_l, hb, blk)
print(f"induced charge in a uniform field: {Q_u:+.2e} C/m   (zero by symmetry)")
print(f"induced charge near a +1e-9 C/m line: {Q_l:+.2e} C/m   (negative, and a fraction of the line charge)")
assert abs(Q_u) < 1e-12 and -1e-9 < Q_l < 0

## 4. Verify

The field at a conductor surface is normal to it. Take the grounded block in the uniform field and measure the angle between $\mathbf{E}$ and the local surface normal on the ring of free cells just outside the block.

Two places have to be excluded, and the reasons are physics, not numerics. On the two faces parallel to the applied field the surface charge changes sign at the midpoint, so $\mathbf{E}$ passes through zero there and its direction is undefined. Near the corners the field wraps around the edge, and a sample one cell out already mixes in that tangential part. So the check skips the three cells nearest each corner and keeps only points where $|\mathbf{E}|$ is at least half of its maximum on the ring, which leaves the two faces normal to the applied field.

In [ ]:
E_u = -fields.gradient(V_u, hb, hb)
a, b = blk
samples = []                                                   # (|E|, angle to the outward normal)
for j in range(a + 3, b - 3):                                  # left and right faces, normals -x and +x
    for i, nx_ in ((a - 1, -1.0), (b, 1.0)):
        e = E_u[i, j, :2]; samples.append((np.linalg.norm(e), np.degrees(np.arccos(abs(e[0] * nx_) / (np.linalg.norm(e) + 1e-300)))))
for i in range(a + 3, b - 3):                                  # bottom and top faces, normals -y and +y
    for j, ny_ in ((a - 1, -1.0), (b, 1.0)):
        e = E_u[i, j, :2]; samples.append((np.linalg.norm(e), np.degrees(np.arccos(abs(e[1] * ny_) / (np.linalg.norm(e) + 1e-300)))))
mag, ang = np.array(samples).T
strong = mag > 0.5 * mag.max()
plt.scatter(mag / mag.max(), ang, s=10); plt.axvline(0.5, color="C3", ls="--")
plt.xlabel("|E| / max |E| on the ring"); plt.ylabel("angle to surface normal [deg]"); plt.yscale("log"); plt.show()
print(f"where |E| >= half its maximum: max angle {ang[strong].max():.2f} deg, median {np.median(ang[strong]):.2f} deg")
print(f"where the field is weak:       max angle {ang[~strong].max():.1f} deg  (direction undefined as |E| -> 0)")
assert ang[strong].max() < 5.0

## 5. Explore and rebuild

Explore. Build a grounded circular conductor (a disk of fixed nodes) next to a line charge. The 2D image for a line charge and a grounded cylinder of radius $R$ is a line charge $-\lambda$ at distance $R^2/d$ from the centre. Confirm it with the solver, then move the line charge in until $d \to R$ and watch where the induced charge concentrates.

Rebuild. Close this notebook and set up the grounded-plane problem from a blank editor: boundary values from the image formula, a single-cell source, the solver call, and the surface charge check.